# Day 7 Lab 11: Incremental Batch Processing

        **Layer:** Cross-layer  
        **Python reference:** `Lab_Files/labs/lab_11_incremental_batch_processing.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Process raw event files one batch at a time.
- Persist a manifest of processed files.
- Run the Medallion flow against the incremental lake path.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


## 1. Import Lab Helpers and Manifest Utility

In [ ]:
import json
from pathlib import Path

from pyspark.sql import functions as F

import importlib
import day7_common
day7_common = importlib.reload(day7_common)


from day7_common import LAKE_DIR, ORDER_SOURCE_FILES, OUTPUT_DIR, STATE_DIR, cleaned_orders, deduplicate_order_events, enriched_orders, ensure_output_dirs, gold_frames, latest_order_state, quality_checked_orders, read_order_events, require_source_data, spark_session, with_bronze_metadata, write_csv_dir, write_json_report

def load_manifest(path: Path) -> dict[str, object]:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {"processed_files": [], "batches": []}

## 2. Start Spark and Load Incremental Manifest

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook11IncrementalProcessing")

incremental_lake = LAKE_DIR / "incremental"
incremental_lake.mkdir(parents=True, exist_ok=True)
manifest_path = STATE_DIR / "notebook_11_incremental_manifest.json"

manifest = load_manifest(manifest_path)
bronze_path = incremental_lake / "bronze" / "orders_raw"
processed_this_run = 0

## 3. Process Each Source File Once

In [ ]:
for index, source_file in enumerate(ORDER_SOURCE_FILES, start=1):
    processed_files = set(manifest["processed_files"])
    if source_file.name in processed_files:
        print(f"Skipping already processed file: {source_file.name}")
        continue

    batch_id = f"notebook-incremental-batch-{index:02d}"
    batch = with_bronze_metadata(read_order_events(spark, [source_file]), batch_id)
    batch.write.mode("append").parquet(str(bronze_path))
    row_count = batch.count()
    manifest["processed_files"].append(source_file.name)
    manifest["batches"].append({"batch_id": batch_id, "source_file": source_file.name, "rows": row_count})
    write_json_report(manifest_path, manifest)
    processed_this_run += 1
    print(f"Processed {source_file.name}: {row_count} Bronze rows")

## 4. Inspect the Manifest

In [ ]:
print(json.dumps(manifest, indent=2))

## 5. Run Silver Transformations

In [ ]:
if not bronze_path.exists():
    raise FileNotFoundError("No incremental Bronze data found. Delete the stale manifest and rerun.")

bronze = spark.read.parquet(str(bronze_path))
checked = quality_checked_orders(cleaned_orders(bronze))
valid = checked.filter(F.col("is_valid"))
deduplicated = deduplicate_order_events(valid)
current = latest_order_state(deduplicated)
enriched = enriched_orders(spark, current)

## 6. Write Incremental Silver and Gold

In [ ]:
valid.write.mode("overwrite").parquet(str(incremental_lake / "silver" / "orders_valid"))
deduplicated.write.mode("overwrite").parquet(str(incremental_lake / "silver" / "orders_deduplicated"))
current.write.mode("overwrite").parquet(str(incremental_lake / "silver" / "orders_current"))
enriched.write.mode("overwrite").parquet(str(incremental_lake / "silver" / "orders_enriched"))

for name, frame in gold_frames(enriched).items():
    frame.write.mode("overwrite").parquet(str(incremental_lake / "gold" / name))

## 7. Build and Write Summary

In [ ]:
summary = spark.createDataFrame(
    [
        ("bronze_rows", bronze.count()),
        ("silver_valid_rows", valid.count()),
        ("silver_deduplicated_rows", deduplicated.count()),
        ("current_order_rows", current.count()),
        ("processed_files", len(manifest["processed_files"])),
        ("files_processed_this_run", processed_this_run),
    ],
    ["metric", "value"],
)
summary.show(truncate=False)
write_csv_dir(summary, OUTPUT_DIR / "notebook_11_incremental_summary")
print(f"Manifest: {manifest_path}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()